In [1]:
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, TimeDistributed, LSTM, Reshape, Dropout

In [ ]:
FOLDER_AUDIO_POTONGAN = "../Data/segmented_audio"
FOLDER_GAMBAR_SPECTRO = "../Data/spectrograms"
os.makedirs(FOLDER_GAMBAR_SPECTRO, exist_ok=True)

In [ ]:
# Otomatis mengambil folder yang ada isinya (6 genre hasil kerja Orang 1)
daftar_genre = [g for g in os.listdir(FOLDER_AUDIO_POTONGAN) if not g.startswith('.')]

print("--- MENGONVERSI AUDIO MENJADI GAMBAR MEL-SPECTROGRAM ---")
for genre in daftar_genre:
    path_in = os.path.join(FOLDER_AUDIO_POTONGAN, genre)
    path_out = os.path.join(FOLDER_GAMBAR_SPECTRO, genre)
    os.makedirs(path_out, exist_ok=True)
    
    for file in tqdm(os.listdir(path_in), desc=f"Genre {genre}"):
        if not file.endswith('.wav'): continue
        path_img = os.path.join(path_out, file.replace('.wav', '.png'))
        if os.path.exists(path_img): continue
            
        try:
            y, sr = librosa.load(os.path.join(path_in, file), sr=None)
            S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
            S_dB = librosa.power_to_db(S, ref=np.max)
            
            fig = plt.figure(figsize=(2.56, 2.56), dpi=100)
            plt.axis('off')
            librosa.display.specshow(S_dB, sr=sr)
            plt.savefig(path_img, bbox_inches='tight', pad_inches=0)
            plt.close(fig)
        except:
            pass

In [ ]:
# Load Dataset Gambar Ke Keras
UKURAN_GAMBAR = (128, 128)
batch = 32
ds_train = tf.keras.utils.image_dataset_from_directory(
    FOLDER_GAMBAR_SPECTRO, validation_split=0.2, subset="training", seed=123, image_size=UKURAN_GAMBAR, batch_size=batch
)
ds_val = tf.keras.utils.image_dataset_from_directory(
    FOLDER_GAMBAR_SPECTRO, validation_split=0.2, subset="validation", seed=123, image_size=UKURAN_GAMBAR, batch_size=batch
)

total_kelas = len(daftar_genre)

In [ ]:
# --- 1. ARSITEKTUR CNN ---
print("\n--- TRAINING ARSITEKTUR MODEL CNN ---")
model_cnn = Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(128, 128, 3)),
    Conv2D(32, (3,3), activation='relu'), MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'), MaxPooling2D((2,2)),
    Flatten(), Dense(128, activation='relu'), Dropout(0.3),
    Dense(total_kelas, activation='softmax')
])
model_cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_cnn.fit(ds_train, validation_data=ds_val, epochs=5) 
model_cnn.save("../models/cnn_genre_model.h5")

In [ ]:
# --- 2. ARSITEKTUR CNN-LSTM ---
print("\n--- TRAINING ARSITEKTUR MODEL HYBRID CNN-LSTM ---")
model_cnn_lstm = Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(128, 128, 3)),
    Reshape((16, 128 * 8, 3)),
    TimeDistributed(Conv2D(32, (3,3), activation='relu', padding='same')),
    TimeDistributed(MaxPooling2D((2,2), padding='same')),
    TimeDistributed(Flatten()),
    LSTM(64, return_sequences=False),
    Dense(64, activation='relu'),
    Dense(total_kelas, activation='softmax')
])
model_cnn_lstm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_cnn_lstm.fit(ds_train, validation_data=ds_val, epochs=5)
model_cnn_lstm.save("../models/cnn_lstm_genre_model.h5")

print("\n[SUKSES]")